In [55]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, List, Dict
from dotenv import load_dotenv
load_dotenv()
import json
from logger import CustomLogger
logger = CustomLogger()

In [12]:
class DataState(TypedDict):
    dataset: Dict
    records: List

In [5]:
llm = init_chat_model(
    "gpt-5-mini-2025-08-07",
    temperature=0
)

In [47]:
SMART_TEST_PROMPT = """
You are a senior QA engineer.

Given this field schema:
{schema}

Generate:
1. Valid cases
2. Boundary cases
3. Invalid cases

Return ONLY JSON:
{
  "valid": [],
  "boundary": [],
  "invalid": []
}
"""
prompt = PromptTemplate.from_template(SMART_TEST_PROMPT)

def llm_generate_cases(field):
    response = llm.invoke(
        prompt.format(schema=json.dumps(field))
    )
    print(json.loads(response.content))
    return json.loads(response.content)

In [49]:
def smart_field_expander(state:DataState):
    new_fields = []
    dataset = state["dataset"]
    for field in ["fields"]:
        cases = llm_generate_cases(field)
        print(cases)
        new_fields.append({
            "name": field["name"],
            "values": (
                cases["valid"] +
                cases["boundary"] +
                cases["invalid"]
            )
        })

    dataset["fields"] = new_fields
    return {"dataset": dataset}

In [50]:
template={
  "dataset_config": {
    "dataset_name": "smart_user_data",
    "type": "smart",
    "record_count": 100
  },
  "fields": [
    {
      "name": "age",
      "type": "integer",
      "min": 18,
      "max": 60
    },
    {
      "name": "email",
      "type": "string",
      "format": "email"
    }
  ]
}

In [51]:
builder = StateGraph(DataState)
builder.add_node("smart_expand", smart_field_expander)

builder.set_entry_point("smart_expand")
builder.add_edge("smart_expand", END)

agent = builder.compile()

In [52]:
response = agent.invoke({"dataset": template})

KeyError: '\n  "valid"'